# Linear Genetic Programming Based Approach for Robotic Controllers
In this lab session, we will leverage Linear Genetic Programming (LGP) to automatically design the controller for a mobile robot. Starting from a certain initial position (S), the robot must be able to navigate inside a maze until it reaches its home base (G). The maze consists of empty cells in which the robot can navigate freely and wall cells where the robot cannot pass. A ```Maze``` object also contains information about the optimal path to the goal, which can be used to evaluate the robot's navigation. 

In [126]:
import enum
import math
import random
import sys

sys.path.insert(0, "./..")

from robot_maze import Maze, Robot

In [127]:
cellcodes = enum.Enum('cellcodes', 'EMPTY WALL START ROUTE GOAL')

#build a maze
maze_list = [
        [cellcodes.EMPTY, cellcodes.EMPTY, cellcodes.EMPTY, cellcodes.EMPTY, cellcodes.WALL, cellcodes.EMPTY, cellcodes.WALL, cellcodes.EMPTY, cellcodes.WALL],
        [cellcodes.WALL, cellcodes.EMPTY, cellcodes.WALL, cellcodes.WALL, cellcodes.WALL, cellcodes.EMPTY, cellcodes.START, cellcodes.EMPTY, cellcodes.WALL],
        [cellcodes.WALL, cellcodes.EMPTY, cellcodes.EMPTY, cellcodes.WALL, cellcodes.ROUTE, cellcodes.ROUTE, cellcodes.ROUTE, cellcodes.EMPTY, cellcodes.WALL],
        [cellcodes.ROUTE, cellcodes.ROUTE, cellcodes.ROUTE, cellcodes.WALL, cellcodes.ROUTE, cellcodes.WALL, cellcodes.WALL, cellcodes.EMPTY, cellcodes.WALL],
        [cellcodes.ROUTE, cellcodes.WALL, cellcodes.ROUTE, cellcodes.ROUTE, cellcodes.ROUTE, cellcodes.WALL, cellcodes.WALL, cellcodes.EMPTY, cellcodes.EMPTY],
        [cellcodes.ROUTE, cellcodes.ROUTE, cellcodes.WALL, cellcodes.WALL, cellcodes.WALL, cellcodes.WALL, cellcodes.WALL, cellcodes.WALL, cellcodes.WALL],
        [cellcodes.WALL, cellcodes.ROUTE, cellcodes.EMPTY, cellcodes.WALL, cellcodes.ROUTE, cellcodes.ROUTE, cellcodes.ROUTE, cellcodes.ROUTE, cellcodes.ROUTE],
        [cellcodes.EMPTY, cellcodes.ROUTE, cellcodes.WALL, cellcodes.WALL, cellcodes.ROUTE, cellcodes.WALL, cellcodes.EMPTY, cellcodes.WALL, cellcodes.ROUTE],
        [cellcodes.WALL, cellcodes.ROUTE, cellcodes.ROUTE, cellcodes.ROUTE, cellcodes.ROUTE, cellcodes.WALL, cellcodes.WALL, cellcodes.WALL, cellcodes.GOAL],
    ]

maze = Maze(maze_list, cellcodes)

print(maze)

            #     #     #  
#     #  #  #     S     #  
#        #  .  .  .     #  
.  .  .  #  .  #  #     #  
.  #  .  .  .  #  #        
.  .  #  #  #  #  #  #  #  
#  .     #  .  .  .  .  .  
   .  #  #  .  #     #  .  
#  .  .  .  .  #  #  #  G  



The robot can move in 3 possible ways:

- Move forward;
- Turn left;
- Turn right;

It can perceive the environment through 6 sensors that, when activated (value 1), indicate the presence of a wall in the corresponding position.

![Robot's sensors](../img/sensors.png "Robot's sensors")

Our controller must map the boolean values coming from the sensors into an action for the robot to take.

Now it's time to code! Define a set of operators which can be included in the program that controls the robot actions.

In [128]:
movecodes = enum.Enum('movecodes', 'FORWARD LEFT RIGHT')
opcodes = enum.Enum('opcodes', 'AND IF NOT OR NOP')

In [129]:
# Generate a random program of length n
# You can tune the probability to have an actual move at a certain position with move_p
def random_program(n, move_p=0.3):
  prg = []
  moves = list(movecodes)
  func = list(opcodes)
  for _ in range(n):
    if random.random() < move_p:
      op = random.choice(moves)
    else:
      op = random.choice(func)
    prg.append(op)
  return prg

In [130]:
rp = random_program(10)

Complete the ```Robot``` class into the ```utilities/robot_maze.py``` file with a proper ```eval``` function.

In [131]:
r = Robot(rp, maze, maxMoves=70, movecodes=movecodes, opcodes=opcodes)

Define a fitness function for our navigation task. Hint: you can exploit the ```getRoute``` method of the ```Robot``` class and the ```scoreRoute``` method of the ```Maze``` class to get an estimate of the "correct steps" taken by the robot.

In [132]:
def fit(prg):
  try:
    r = Robot(prg, maze, maxMoves=70, movecodes=movecodes, opcodes=opcodes)
    r.run()
    fitness = maze.scoreRoute(r.getRoute()) - r.n_moves
    return fitness
  except:
    return -math.inf

As usual, the selection strategy we choose is tournament selection.

In [133]:
def tournament_selection(fit, pop, t_size=4):
  tournament = random.choices(pop, k=t_size)
  return max(tournament, key=fit)

Implement functions for crossover and mutation.

In [134]:
def crossover(x, y):
  # random indexes over the possible indexes of the individuals
  psx = sorted(random.choices([i for i in range (1, len(x))], k=2))
  psy = sorted(random.choices([i for i in range (1, len(y))], k=2))
  # offspring 1 
  offspring1 = x[:psx[0]]
  offspring1.extend(y[psy[0]:psy[1]])
  offspring1.extend(x[psx[1]:])
  # offspring 2
  offspring2 = y[:psy[0]]
  offspring2.extend(x[psx[0]:psx[1]])
  offspring2.extend(y[psy[1]:])

  return offspring1, offspring2

In [135]:
x = random_program(10)
y = random_program(10)
print(x)
print(y)
z, w= crossover(x, y)
print(z)
print(w)

[<movecodes.RIGHT: 3>, <opcodes.NOT: 3>, <opcodes.OR: 4>, <opcodes.NOT: 3>, <movecodes.FORWARD: 1>, <movecodes.FORWARD: 1>, <movecodes.FORWARD: 1>, <movecodes.RIGHT: 3>, <movecodes.RIGHT: 3>, <opcodes.IF: 2>]
[<movecodes.LEFT: 2>, <opcodes.IF: 2>, <opcodes.NOT: 3>, <opcodes.IF: 2>, <opcodes.AND: 1>, <opcodes.NOT: 3>, <opcodes.NOP: 5>, <opcodes.NOT: 3>, <movecodes.FORWARD: 1>, <opcodes.AND: 1>]
[<movecodes.RIGHT: 3>, <opcodes.NOP: 5>, <opcodes.NOT: 3>, <movecodes.FORWARD: 1>, <movecodes.RIGHT: 3>, <movecodes.RIGHT: 3>, <opcodes.IF: 2>]
[<movecodes.LEFT: 2>, <opcodes.IF: 2>, <opcodes.NOT: 3>, <opcodes.IF: 2>, <opcodes.AND: 1>, <opcodes.NOT: 3>, <opcodes.NOT: 3>, <opcodes.OR: 4>, <opcodes.NOT: 3>, <movecodes.FORWARD: 1>, <movecodes.FORWARD: 1>, <movecodes.FORWARD: 1>, <opcodes.AND: 1>]


In [136]:
def mutation(x, p_m, move_p=0.3):
  for i in range (len(x)):
    if random.uniform(0, 1) < p_m:
      if random.uniform(0, 1) < move_p:
        x[i] = random.choice(list(movecodes))
      else:
        x[i] = random.choice(list(opcodes))
  return x

Finally, we can implement a ```linear_GP``` function, using the functions defined above.

In [137]:
def linear_GP(fit, pop_size, n_iter = 50):
  p_m = 0.1
  pop = [random_program(20) for _ in range(0, pop_size)]
  best = []
  for i in range(0, n_iter):
    selected = [tournament_selection(fit, pop) for _ in range(0, pop_size)]
    pairs = zip(selected, selected[1:] + [selected[0]])
    offsprings = []
    for x, y in pairs:
      of1, of2 = crossover(x, y)
      offsprings.append(of1)
      offsprings.append(of2)
    pop = [mutation(x, p_m) for x in offsprings]
    candidate_best = max(pop, key=fit)
    if fit(candidate_best) > fit(best):
      best = candidate_best
    # print(f"Best individual at generation {i}: {best}")
    print(f"Best fitness at generation {i}: {fit(best)}")
  return best

In [138]:
random.seed(0)
best = linear_GP(fit, 1000)

Best fitness at generation 0: -67
Best fitness at generation 1: -67
Best fitness at generation 2: -67
Best fitness at generation 3: -67
Best fitness at generation 4: -67
Best fitness at generation 5: -67
Best fitness at generation 6: -67
Best fitness at generation 7: -67
Best fitness at generation 8: -67
Best fitness at generation 9: -67
Best fitness at generation 10: -67
Best fitness at generation 11: -67
Best fitness at generation 12: -67
Best fitness at generation 13: -67
Best fitness at generation 14: -67
Best fitness at generation 15: -67
Best fitness at generation 16: -67
Best fitness at generation 17: -67
Best fitness at generation 18: -67
Best fitness at generation 19: -67
Best fitness at generation 20: -67
Best fitness at generation 21: -67
Best fitness at generation 22: -65
Best fitness at generation 23: -65
Best fitness at generation 24: -65
Best fitness at generation 25: -65
Best fitness at generation 26: -65
Best fitness at generation 27: -65
Best fitness at generation 28:

In [139]:
random.seed(0)
best = linear_GP(fit, 1000)

Best fitness at generation 0: -67
Best fitness at generation 1: -67
Best fitness at generation 2: -67
Best fitness at generation 3: -67
Best fitness at generation 4: -67
Best fitness at generation 5: -67
Best fitness at generation 6: -67
Best fitness at generation 7: -67
Best fitness at generation 8: -67
Best fitness at generation 9: -67
Best fitness at generation 10: -67
Best fitness at generation 11: -67
Best fitness at generation 12: -67
Best fitness at generation 13: -67
Best fitness at generation 14: -67
Best fitness at generation 15: -67
Best fitness at generation 16: -67
Best fitness at generation 17: -67
Best fitness at generation 18: -67
Best fitness at generation 19: -67
Best fitness at generation 20: -67
Best fitness at generation 21: -67
Best fitness at generation 22: -65
Best fitness at generation 23: -65
Best fitness at generation 24: -65
Best fitness at generation 25: -65
Best fitness at generation 26: -65
Best fitness at generation 27: -65
Best fitness at generation 28:

In [144]:
best_robot = Robot(best, maze, 70, movecodes, opcodes)
best_robot.run()
print(best_robot.getRoute())

[[6, 1], [6, 2], [7, 2], [6, 2], [5, 2], [4, 2], [4, 3], [4, 4]]


Are the robot moves consistent with the expected behaviour? (The robot starts with a south heading, hence towards the bottom of the monitor)

In [145]:
print(maze)
best_robot.moves

            #     #     #  
#     #  #  #     S     #  
#        #  .  .  .     #  
.  .  .  #  .  #  #     #  
.  #  .  .  .  #  #        
.  .  #  #  #  #  #  #  #  
#  .     #  .  .  .  .  .  
   .  #  #  .  #     #  .  
#  .  .  .  .  #  #  #  G  



[<movecodes.FORWARD: 1>,
 <movecodes.LEFT: 2>,
 <movecodes.FORWARD: 1>,
 <movecodes.LEFT: 2>,
 <movecodes.LEFT: 2>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.LEFT: 2>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.FORWARD: 1>,
 <movecodes.